# COM837 Assignment 3 - ChannelNet: full pipeline (overnight run)
Runs everything from scratch: conventional baselines, SRCNN + DnCNN training (12 & 22 dB, both channel models), pilot-density sweep, final evaluation. Outputs: `/kaggle/working/results`, `/kaggle/working/checkpoints`, and `submission_outputs.zip`.

In [1]:
# 0. Copy code into the working dir and locate the channel data
import glob, os, shutil
code_dir = os.path.dirname(glob.glob('/kaggle/input/**/run_conventional.py', recursive=True)[0])
shutil.copytree(code_dir, '/kaggle/working', dirs_exist_ok=True)
data_dir = os.path.dirname(glob.glob('/kaggle/input/**/H_VehA.mat', recursive=True)[0])
os.environ['MLWC_DATA'] = data_dir
os.chdir('/kaggle/working')
print('code:', code_dir, '\ndata:', data_dir)
!nvidia-smi --query-gpu=name,memory.total --format=csv

code: /kaggle/input/datasets/dhavala579/mlwl-a3-code 
data: /kaggle/input/datasets/dhavala579/mlwl-a3-channels
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
# 1. Task 1 + 2: visualisations and conventional estimators (LS, MMSE)
!python run_conventional.py

[VehA] train (32000, 72, 14), test (4000, 72, 14)
  SNR  0 dB  LS 6.18e-01  MMSE-ideal 7.15e-02  MMSE-est 1.04e-01
  SNR  5 dB  LS 1.97e-01  MMSE-ideal 3.19e-02  MMSE-est 4.31e-02
  SNR 10 dB  LS 6.34e-02  MMSE-ideal 1.28e-02  MMSE-est 1.72e-02
  SNR 15 dB  LS 2.12e-02  MMSE-ideal 4.89e-03  MMSE-est 7.21e-03
  SNR 20 dB  LS 7.84e-03  MMSE-ideal 1.84e-03  MMSE-est 3.69e-03
  SNR 25 dB  LS 3.61e-03  MMSE-ideal 6.72e-04  MMSE-est 2.39e-03
  SNR 30 dB  LS 2.27e-03  MMSE-ideal 2.43e-04  MMSE-est 1.91e-03
  pilot sweep @ 12 dB: {'snr_db': 12, 'pilots': [24, 36, 48, 72], 'LS': [0.04568302631378174, 0.041897617280483246, 0.04054582118988037, 0.040550243109464645], 'MMSE_ideal': [0.015812065452337265, 0.01117690373212099, 0.008789991959929466, 0.006405889987945557]}
[VehA] done in 11 s
[SUI5] train (32000, 72, 14), test (4000, 72, 14)
  SNR  0 dB  LS 7.46e-01  MMSE-ideal 6.71e-02  MMSE-est 2.01e-01
  SNR  5 dB  LS 3.24e-01  MMSE-ideal 2.69e-02  MMSE-est 1.55e-01
  SNR 10 dB  LS 1.91e-01  MMSE-i

In [3]:
# 2. Task 3: SRCNN super-resolution stage, 48 pilots, trained at 12 and 22 dB
for m in ['VehA', 'SUI5']:
    for s in [12, 22]:
        !python train.py --model {m} --snr {s} --pilots 48 --stage sr

device cuda, GPUs 2, AMP True, tag VehA_snr12_p48
train images (64000, 1, 72, 14), val (8000, 1, 72, 14), scale 0.7075
SRCNN parameters: 8129
input (LS+interp) val loss before training: 4.0553e-02
[VehA_snr12_p48_sr] ep   1  train 3.4294e-02  val 1.6894e-02  lr 1.0e-03  7.3s *
[VehA_snr12_p48_sr] ep   2  train 1.6588e-02  val 1.6554e-02  lr 1.0e-03  2.6s *
[VehA_snr12_p48_sr] ep   3  train 1.6076e-02  val 1.5836e-02  lr 1.0e-03  2.6s *
[VehA_snr12_p48_sr] ep   4  train 1.5816e-02  val 1.5911e-02  lr 1.0e-03  2.6s 
[VehA_snr12_p48_sr] ep   5  train 1.5726e-02  val 1.5591e-02  lr 1.0e-03  2.6s *
[VehA_snr12_p48_sr] ep   6  train 1.5640e-02  val 1.5398e-02  lr 1.0e-03  2.6s *
[VehA_snr12_p48_sr] ep   7  train 1.5564e-02  val 1.5737e-02  lr 1.0e-03  2.6s 
[VehA_snr12_p48_sr] ep   8  train 1.5452e-02  val 1.5478e-02  lr 1.0e-03  2.6s 
[VehA_snr12_p48_sr] ep   9  train 1.5427e-02  val 1.5305e-02  lr 1.0e-03  2.6s *
[VehA_snr12_p48_sr] ep  10  train 1.5408e-02  val 1.5932e-02  lr 1.0e-03  2.7

In [4]:
# 3. Task 4/5: DnCNN restoration stage on top of the frozen SRCNN
for m in ['VehA', 'SUI5']:
    for s in [12, 22]:
        !python train.py --model {m} --snr {s} --pilots 48 --stage dn --epochs_dn 40

device cuda, GPUs 2, AMP True, tag VehA_snr12_p48
train images (64000, 1, 72, 14), val (8000, 1, 72, 14), scale 0.7075
DnCNN parameters: 667073
SR-only val loss: 1.4669e-02
[VehA_snr12_p48_dn] ep   1  train 1.7470e-02  val 1.4519e-02  lr 1.0e-03  27.7s *
[VehA_snr12_p48_dn] ep   2  train 1.4024e-02  val 1.3362e-02  lr 1.0e-03  25.5s *
[VehA_snr12_p48_dn] ep   3  train 1.3094e-02  val 1.3309e-02  lr 1.0e-03  25.1s *
[VehA_snr12_p48_dn] ep   4  train 1.2812e-02  val 1.2837e-02  lr 1.0e-03  25.1s *
[VehA_snr12_p48_dn] ep   5  train 1.2684e-02  val 1.4298e-02  lr 1.0e-03  25.3s 
[VehA_snr12_p48_dn] ep   6  train 1.2631e-02  val 1.2714e-02  lr 1.0e-03  25.3s *
[VehA_snr12_p48_dn] ep   7  train 1.2537e-02  val 1.3064e-02  lr 1.0e-03  25.2s 
[VehA_snr12_p48_dn] ep   8  train 1.2511e-02  val 1.2583e-02  lr 1.0e-03  25.3s *
[VehA_snr12_p48_dn] ep   9  train 1.2484e-02  val 1.3112e-02  lr 1.0e-03  25.4s 
[VehA_snr12_p48_dn] ep  10  train 1.2411e-02  val 1.2437e-02  lr 1.0e-03  25.5s *
[VehA_snr1

In [5]:
# 4. Pilot-density sweep (Fig. 5): full ChannelNet at 12 dB for 24, 36, 72 pilots
for m in ['VehA', 'SUI5']:
    for p in [24, 36, 72]:
        !python train.py --model {m} --snr 12 --pilots {p} --stage both --epochs_dn 20

device cuda, GPUs 2, AMP True, tag VehA_snr12_p24
train images (64000, 1, 72, 14), val (8000, 1, 72, 14), scale 0.7075
SRCNN parameters: 8129
input (LS+interp) val loss before training: 4.5642e-02
[VehA_snr12_p24_sr] ep   1  train 4.3659e-02  val 2.6116e-02  lr 1.0e-03  3.5s *
[VehA_snr12_p24_sr] ep   2  train 2.5945e-02  val 2.5679e-02  lr 1.0e-03  2.7s *
[VehA_snr12_p24_sr] ep   3  train 2.5522e-02  val 2.5309e-02  lr 1.0e-03  2.7s *
[VehA_snr12_p24_sr] ep   4  train 2.5445e-02  val 2.5995e-02  lr 1.0e-03  2.8s 
[VehA_snr12_p24_sr] ep   5  train 2.5376e-02  val 2.5040e-02  lr 1.0e-03  2.8s *
[VehA_snr12_p24_sr] ep   6  train 2.5206e-02  val 2.5318e-02  lr 1.0e-03  2.8s 
[VehA_snr12_p24_sr] ep   7  train 2.5155e-02  val 2.5014e-02  lr 1.0e-03  2.8s *
[VehA_snr12_p24_sr] ep   8  train 2.5106e-02  val 2.5268e-02  lr 1.0e-03  2.8s 
[VehA_snr12_p24_sr] ep   9  train 2.5029e-02  val 2.5270e-02  lr 1.0e-03  2.8s 
[VehA_snr12_p24_sr] ep  10  train 2.4953e-02  val 2.4808e-02  lr 1.0e-03  2.8s

In [6]:
# 5. Task 6: evaluation from saved checkpoints -> Fig. 3, 4, 5 and tables
!python test.py

[VehA] SNR  0.0: LS 6.18e-01  MMSE_est 1.04e-01  MMSE_ideal 7.15e-02  SR_12dB 1.78e-01  ChannelNet_12dB 1.66e-01  SR_22dB 3.96e-01  ChannelNet_22dB 7.62e-01  ChannelNet_switched 1.66e-01
[VehA] SNR  2.5: LS 3.50e-01  MMSE_est 6.74e-02  MMSE_ideal 4.81e-02  SR_12dB 9.99e-02  ChannelNet_12dB 9.15e-02  SR_22dB 2.16e-01  ChannelNet_22dB 2.84e-01  ChannelNet_switched 9.15e-02
[VehA] SNR  5.0: LS 1.97e-01  MMSE_est 4.21e-02  MMSE_ideal 3.19e-02  SR_12dB 5.77e-02  ChannelNet_12dB 5.22e-02  SR_22dB 1.18e-01  ChannelNet_22dB 1.15e-01  ChannelNet_switched 5.22e-02
[VehA] SNR  7.5: LS 1.11e-01  MMSE_est 2.71e-02  MMSE_ideal 2.03e-02  SR_12dB 3.35e-02  ChannelNet_12dB 2.97e-02  SR_22dB 6.43e-02  ChannelNet_22dB 5.31e-02  ChannelNet_switched 2.97e-02
[VehA] SNR 10.0: LS 6.34e-02  MMSE_est 1.72e-02  MMSE_ideal 1.28e-02  SR_12dB 2.06e-02  ChannelNet_12dB 1.78e-02  SR_22dB 3.54e-02  ChannelNet_22dB 2.73e-02  ChannelNet_switched 1.78e-02
[VehA] SNR 12.5: LS 3.64e-02  MMSE_est 1.11e-02  MMSE_ideal 7.95e

In [7]:
# 6. Pack everything for download (Output tab -> submission_outputs.zip)
!zip -r -q submission_outputs.zip results checkpoints
!ls -la results checkpoints
!du -sh submission_outputs.zip

checkpoints:
total 26968
drwxr-xr-x 2 root root    4096 Nov 21 20:28 .
drwxrwxrwx 6 root root    4096 Nov 21 20:34 ..
-rw-r--r-- 1 root root 2715771 Nov 21 20:02 SUI5_snr12_p24_dn.pt
-rw-r--r-- 1 root root   38989 Nov 21 19:53 SUI5_snr12_p24_sr.pt
-rw-r--r-- 1 root root 2715771 Nov 21 20:15 SUI5_snr12_p36_dn.pt
-rw-r--r-- 1 root root   38989 Nov 21 20:07 SUI5_snr12_p36_sr.pt
-rw-r--r-- 1 root root 2716347 Nov 21 18:52 SUI5_snr12_p48_dn.pt
-rw-r--r-- 1 root root   38989 Nov 21 17:59 SUI5_snr12_p48_sr.pt
-rw-r--r-- 1 root root 2715771 Nov 21 20:28 SUI5_snr12_p72_dn.pt
-rw-r--r-- 1 root root   38989 Nov 21 20:20 SUI5_snr12_p72_sr.pt
-rw-r--r-- 1 root root 2716347 Nov 21 19:09 SUI5_snr22_p48_dn.pt
-rw-r--r-- 1 root root   38989 Nov 21 18:03 SUI5_snr22_p48_sr.pt
-rw-r--r-- 1 root root 2715771 Nov 21 19:22 VehA_snr12_p24_dn.pt
-rw-r--r-- 1 root root   38733 Nov 21 19:13 VehA_snr12_p24_sr.pt
-rw-r--r-- 1 root root 2715771 Nov 21 19:35 VehA_snr12_p36_dn.pt
-rw-r--r-- 1 root root   38861 Nov 21